# Stage 2: DNAFormer compatibility check (no training)
**Use a GPU:** Runtime > Change runtime type > **T4 GPU** before running.

Run the cells in order. Nothing to upload.

### Cell 1: mount Drive, set paths, check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/fyp_dna'
CODE    = '/content/drive/MyDrive/fyp_dna/code'
REPO    = '/content/Deep-DNA-based-storage'
DATA    = f'{PROJECT}/data/stage1_pilot'
CKPT_DIR = f'{PROJECT}/checkpoints'
CKPT    = f'{CKPT_DIR}/DNAFormer_siamese.pth'
os.makedirs(CODE, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(f'{PROJECT}/logs', exist_ok=True)
print('stage1_generate.py on Drive:', os.path.isfile(f'{CODE}/stage1_generate.py'))
print('Stage 1 data on Drive:', os.path.isfile(f'{DATA}/config.json'))
!nvidia-smi --query-gpu=name,memory.total --format=csv

### Cell 2: install the two small packages DNAFormer needs

In [ ]:
!pip install -q einops gdown

### Cell 3: write the adapter (reused in later stages)

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/dnaf_adapter.py
"""
Minimal adapter between our Stage 1 data and the UNCHANGED DNAFormer model.

What it does:
  - builds DNAFormer exactly as the repo defines it (DNN/model_DNAFormer_siamese.py)
  - loads the pretrained checkpoint and REFUSES to continue if any weight is missing
    (the repo uses strict=False, which silently leaves layers random)
  - turns a cluster of reads into the model's input tensors, without the repo
    loader's problems (crash on Python >= 3.11, reads of length 132 turned into
    zeros, deletion-heavy reads thrown away, clusters silently swapped)

The model architecture is not touched. Input/target lengths stay 132 / 128.
"""
import os
import sys

import numpy as np

BASES = "ACGT"
BASE_IDX = {b: i for i, b in enumerate(BASES)}   # same mapping as the repo: A=0 C=1 G=2 T=3

LABEL_LEN = 128        # repo: label_length 140 - index_length 12
INPUT_LEN = 132        # repo: noisy_copies_length = 140 + 4 - 12
MAX_READS = 16         # repo: max_number_per_cluster
ORIG_MAX_DEVIATION = 4  # repo: corrupt_max_deviation


class DNAFormerConfig:
    """Same values as DNN/inference.py in the repo."""
    index_length = 12
    label_length = 140
    filter_index = True
    corrupt_max_deviation = 4
    noisy_copies_length = INPUT_LEN
    max_number_per_cluster = MAX_READS
    read_padding = "end"
    model_config = "siamese"
    n_head = 32
    activation = "gelu"
    num_layers = 12
    d_model = 1024
    alignment_filters = 128
    alignement_filters = 128   # the repo's train.py uses this misspelling
    dim_feedforward = 2048
    output_ch = 4
    enc_filters = 4
    p_dropout = 0
    class_token = 0
    use_input_scaling = False
    device = "cpu"


# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------
def build_model(repo, device="cpu"):
    sys.path.insert(0, os.path.join(repo, "DNN"))
    from model_DNAFormer_siamese import net  # noqa: E402
    cfg = DNAFormerConfig()
    cfg.device = device
    return net(cfg).to(device), cfg


def load_checkpoint(model, path, device="cpu"):
    """Load weights strictly. Returns a report dict; raises if anything does not match."""
    import torch
    try:
        ckpt = torch.load(path, map_location=device, weights_only=True)
        loader = "weights_only=True"
    except Exception:
        # Official checkpoint from the DNAFormer authors; may contain non-tensor objects.
        ckpt = torch.load(path, map_location=device, weights_only=False)
        loader = "weights_only=False"
    state = ckpt.get("model_state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    state = {k[7:] if k.startswith("module.") else k: v for k, v in state.items()}
    model_keys = set(model.state_dict().keys())
    ckpt_keys = set(state.keys())
    missing = sorted(model_keys - ckpt_keys)
    unexpected = sorted(ckpt_keys - model_keys)
    shape_bad = sorted(k for k in model_keys & ckpt_keys
                       if tuple(model.state_dict()[k].shape) != tuple(state[k].shape))
    report = {
        "loader": loader,
        "checkpoint_top_keys": sorted(ckpt.keys()) if isinstance(ckpt, dict) else [],
        "checkpoint_epoch": ckpt.get("epoch") if isinstance(ckpt, dict) else None,
        "n_tensors": len(ckpt_keys), "missing": missing, "unexpected": unexpected,
        "shape_mismatch": shape_bad,
    }
    if missing or unexpected or shape_bad:
        raise RuntimeError("Checkpoint does not match the model: %s" % report)
    model.load_state_dict(state, strict=True)
    return report


# ---------------------------------------------------------------------------
# Input encoding
# ---------------------------------------------------------------------------
def select_reads(reads, policy="relaxed"):
    """Which reads go into the model.

    relaxed  (ours): keep every read; reads longer than 132 are cut to 132.
    original (repo): keep reads with |len-128| <= 4; a read of exactly 132 becomes
                     an all-zero row (repo bug). If no read survives, the repo would
                     replace the whole cluster with a random other one.
    """
    if policy == "relaxed":
        return [r[:INPUT_LEN] for r in reads[:MAX_READS]]
    if policy == "original":
        kept = [r for r in reads if abs(len(r) - LABEL_LEN) <= ORIG_MAX_DEVIATION]
        return [r if len(r) < INPUT_LEN else "" for r in kept[:MAX_READS]]
    raise ValueError(policy)


def one_hot(read, length=INPUT_LEN):
    x = np.zeros((4, length), dtype=np.float32)
    for j, b in enumerate(read[:length]):
        x[BASE_IDX[b], j] = 1.0
    return x


def encode_cluster(reads, policy="relaxed"):
    """Return (left, right, n_used) as numpy arrays of shape [16, 4, 132].

    left  = each read one-hot, zero-padded at the end (repo read_padding='end')
    right = each read reversed, then zero-padded at the end (repo siamese branch)
    Unused read slots stay all-zero, as in the repo.
    """
    used = select_reads(reads, policy)
    left = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    right = np.zeros((MAX_READS, 4, INPUT_LEN), dtype=np.float32)
    for i, r in enumerate(used):
        left[i] = one_hot(r)
        right[i] = one_hot(r[::-1])
    return left, right, len(used)


def make_batch(clusters, policy="relaxed", device="cpu"):
    """clusters: list of Stage 1 metadata dicts. Returns (model_input, labels).

    model_input has shape [2B, 16, 4, 132]: left branch first, right branch second,
    exactly as the repo concatenates them. labels has shape [B, 128] (class indices).
    """
    import torch
    L, R, Y = [], [], []
    for c in clusters:
        left, right, _ = encode_cluster(c["noisy_reads"], policy)
        L.append(left)
        R.append(right)
        Y.append([BASE_IDX[b] for b in c["clean_strand"]])
    x = torch.from_numpy(np.concatenate([np.stack(L), np.stack(R)], axis=0)).to(device)
    y = torch.tensor(Y, dtype=torch.long, device=device)
    return x, y


def decode(pred_logits):
    """[B, 4, 128] logits -> list of DNA strings."""
    idx = pred_logits.argmax(dim=1).cpu().numpy()
    return ["".join(BASES[i] for i in row) for row in idx]


# ---------------------------------------------------------------------------
# Scoring helpers
# ---------------------------------------------------------------------------
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]


### Cell 4: write the Stage 2 check script

In [ ]:
%%writefile /content/drive/MyDrive/fyp_dna/code/stage2_check.py
"""
Stage 2: DNAFormer compatibility check. No training.

Checks that:
  A. the environment and GPU are usable
  B. the pretrained checkpoint loads into the unchanged model with NO missing weights,
     and its layer sizes confirm input 132 / output 128
  C. the repo's own data loader has the problems we expect (so we don't use it)
  D. our adapter turns one marker-free, one periodic and one end-block cluster into
     correctly shaped input
  E. the pretrained model runs on them and gives 128-base output
  F. a small sanity inference: pretrained model on a few test clusters per level
  G. GPU memory and speed for one fine-tuning step at several batch sizes

Usage:
  python stage2_check.py --repo REPO --data DATA --ckpt CKPT [--n-sanity 50] [--log out.json]
  python stage2_check.py --repo REPO --data DATA --no-torch     # only the parts without PyTorch
"""
import argparse
import json
import os
import platform
import random
import sys
import time
import types

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
import dnaf_adapter as A  # noqa: E402
import stage1_generate as G  # noqa: E402

FAILS = []
SUMMARY = {}


def check(ok, msg):
    print(("  [PASS] " if ok else "  [FAIL] ") + msg)
    if not ok:
        FAILS.append(msg)


def load_subset(data, arm, subset):
    with open(os.path.join(data, arm, subset, "metadata.json")) as f:
        return json.load(f)


def score(arm, pred, cluster):
    payload = cluster["payload_strand"]
    pred_payload = G.remove_markers(arm, pred)
    ed = A.edit_distance(pred_payload, payload)
    return ed / len(payload), pred_payload == payload


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--repo", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--n-sanity", type=int, default=50, help="test clusters per level for the sanity inference")
    ap.add_argument("--no-torch", action="store_true")
    ap.add_argument("--log", default=None)
    args = ap.parse_args()
    levels = G.DELETION_LEVELS
    tests = ["test_" + G.level_tag(d) for d in levels]

    # ------------------------------------------------------------------ A
    print("A. Environment")
    print("   python %s" % platform.python_version())
    SUMMARY["python"] = platform.python_version()
    device = "cpu"
    torch = None
    if not args.no_torch:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print("   torch %s, device %s" % (torch.__version__, device))
        SUMMARY["torch"] = torch.__version__
        if device == "cuda":
            p = torch.cuda.get_device_properties(0)
            print("   GPU %s, %.1f GB" % (p.name, p.total_memory / 1e9))
            SUMMARY["gpu"] = p.name
            SUMMARY["gpu_memory_gb"] = round(p.total_memory / 1e9, 1)
        else:
            print("   WARNING: no GPU. Switch Runtime > Change runtime type > T4 GPU.")
        torch.manual_seed(0)
    random.seed(0)

    # ------------------------------------------------------------------ C (no torch needed)
    print("\nC. Problems in the repo's own data loader")
    try:
        random.shuffle([1, 2, 3], random.random)
        shuffle_ok = True
    except TypeError:
        shuffle_ok = False
    print("   repo read_data() calls random.shuffle(x, random.random): %s on Python %s"
          % ("works" if shuffle_ok else "CRASHES (TypeError)", platform.python_version()))
    SUMMARY["repo_shuffle_crashes"] = not shuffle_ok

    if not args.no_torch:
        try:
            sys.path.insert(0, args.repo)
            from DNN import data_loader as repo_dl  # noqa: E402
            fake = types.SimpleNamespace(config=A.DNAFormerConfig())
            reads = ["A" * 132, "C" * 128]
            left, _, _, _ = repo_dl.grab_model_input(fake, {"noisy_copies": reads})
            row132, row128 = float(left[0].sum()), float(left[1].sum())
            print("   repo grab_model_input(): read of length 132 -> %d non-zero bases; length 128 -> %d"
                  % (row132, row128))
            SUMMARY["repo_len132_becomes_zero"] = row132 == 0
        except Exception as e:  # noqa: BLE001
            print("   (could not import repo data_loader to demonstrate: %s)" % e)

    print("   repo length filter (|len-128|<=4) on our test sets; a cluster with 0 reads left is")
    print("   silently REPLACED by a random other cluster in the repo loader:")
    print("   arm              level  reads kept (of 10)  clusters swapped")
    SUMMARY["repo_filter"] = {}
    for arm in ["marker_free", "marker_periodic"]:
        for t, d in zip(tests, levels):
            cl = load_subset(args.data, arm, t)
            kept = [len(A.select_reads(c["noisy_reads"], "original")) for c in cl]
            swapped = float(np.mean([k == 0 for k in kept]))
            print("   %-16s %4.0f%%   %5.2f               %5.1f%%" % (arm, d * 100, np.mean(kept), 100 * swapped))
            SUMMARY["repo_filter"]["%s_%s" % (arm, t)] = {"mean_kept": round(float(np.mean(kept)), 2),
                                                          "swapped_frac": round(swapped, 3)}

    # ------------------------------------------------------------------ D (no torch needed)
    print("\nD. Adapter encoding of one cluster per arm (test_del05, pair 0)")
    examples = {arm: load_subset(args.data, arm, "test_del05")[0] for arm in G.ARMS}
    check(len({c["pair_id"] for c in examples.values()}) == 1, "the three example clusters are the same pair")
    for arm, c in examples.items():
        left, right, n = A.encode_cluster(c["noisy_reads"], "relaxed")
        lens = [len(r) for r in c["noisy_reads"]]
        filled = left.sum(axis=(1,))  # [16, 132], 1 where a base is present
        ok = (left.shape == (16, 4, 132) and right.shape == (16, 4, 132) and n == 10 and
              all(int(filled[i].sum()) == min(lens[i], 132) for i in range(10)) and
              filled[10:].sum() == 0 and
              all(A.one_hot(c["noisy_reads"][i][::-1]).tolist() == right[i].tolist() for i in range(10)))
        check(ok, "%-16s input [16,4,132] x2, 10 reads used, slots 11-16 empty, read lengths %d-%d"
              % (arm, min(lens), max(lens)))
        check(len(c["clean_strand"]) == A.LABEL_LEN, "%-16s target length 128 matches model output" % arm)

    if args.no_torch:
        finish(args)
        return

    # ------------------------------------------------------------------ B
    print("\nB. Pretrained checkpoint")
    model, cfg = A.build_model(args.repo, device)
    n_params = sum(p.numel() for p in model.parameters())
    print("   model built from repo code: %.1fM parameters" % (n_params / 1e6))
    SUMMARY["n_params_M"] = round(n_params / 1e6, 1)
    if not args.ckpt or not os.path.isfile(args.ckpt):
        check(False, "checkpoint file found at %s" % args.ckpt)
        finish(args)
        return
    try:
        rep = A.load_checkpoint(model, args.ckpt, device)
        check(True, "all %d weight tensors loaded strictly (0 missing, 0 unexpected, 0 wrong shape) [%s]"
              % (rep["n_tensors"], rep["loader"]))
        print("   checkpoint keys: %s, epoch: %s" % (rep["checkpoint_top_keys"], rep["checkpoint_epoch"]))
        SUMMARY["checkpoint"] = {k: rep[k] for k in ("loader", "n_tensors", "checkpoint_epoch")}
    except RuntimeError as e:
        check(False, str(e)[:500])
        finish(args)
        return
    sd = model.state_dict()
    shapes = {
        "alignement.linear_block.fc_1.weight": (132, 132),
        "embedding.linear_block.fc_1.weight": (128, 132),
        "fusion.pred_fusion_left": (128,),
    }
    for k, want in shapes.items():
        check(tuple(sd[k].shape) == want, "%s has shape %s (input 132 -> output 128)" % (k, tuple(sd[k].shape)))

    # ------------------------------------------------------------------ E
    print("\nE. Forward pass on the three example clusters (pretrained, no fine-tuning)")
    model.eval()
    for arm, c in examples.items():
        x, y = A.make_batch([c], "relaxed", device)
        with torch.inference_mode():
            out = model(x)
        shp = tuple(out["pred"].shape)
        check(shp == (1, 4, 128), "%-16s output shape %s" % (arm, shp))
        pred = A.decode(out["pred"])[0]
        ned, exact = score(arm, pred, c)
        print("     payload-only normalized edit distance %.3f, exact=%s" % (ned, exact))

    # ------------------------------------------------------------------ F
    print("\nF. Sanity inference: pretrained DNAFormer, %d test clusters per level" % args.n_sanity)
    print("   (only a sanity check; the proper baseline evaluation is Stage 3)")
    print("   arm / read policy            level  exact   norm. edit dist")
    runs = [("marker_free", "original"), ("marker_free", "relaxed"),
            ("marker_periodic", "relaxed"), ("marker_endblock", "relaxed")]
    SUMMARY["sanity"] = {}
    for arm, pol in runs:
        for t, d in zip(tests, levels):
            cl = load_subset(args.data, arm, t)[:args.n_sanity]
            neds, exacts = [], []
            for i in range(0, len(cl), 25):
                chunk = cl[i:i + 25]
                x, _ = A.make_batch(chunk, pol, device)
                with torch.inference_mode():
                    preds = A.decode(model(x)["pred"])
                for p, c in zip(preds, chunk):
                    ned, ex = score(arm, p, c)
                    neds.append(ned)
                    exacts.append(ex)
            print("   %-16s %-9s     %4.0f%%  %5.1f%%  %.4f" % (arm, pol, d * 100, 100 * np.mean(exacts), np.mean(neds)))
            SUMMARY["sanity"]["%s_%s_%s" % (arm, pol, t)] = {"exact": round(float(np.mean(exacts)), 3),
                                                            "ned": round(float(np.mean(neds)), 4)}
    mf1 = SUMMARY["sanity"]["marker_free_relaxed_test_del01"]["ned"]
    check(mf1 < 0.05, "pretrained model is clearly working at 1%% deletion (norm. edit dist %.4f < 0.05)" % mf1)

    # ------------------------------------------------------------------ G
    print("\nG. One fine-tuning step: GPU memory and speed (full model, Adam, fp32)")
    if device != "cuda":
        print("   skipped (no GPU)")
    else:
        train = load_subset(args.data, "marker_free", "train")
        model.train()
        opt = torch.optim.Adam(model.parameters(), lr=1e-5)
        ce = torch.nn.CrossEntropyLoss()
        SUMMARY["memory"] = {}
        for bs in [8, 16, 32, 64]:
            try:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats()
                x, y = A.make_batch(train[:bs], "relaxed", device)
                times = []
                for _ in range(3):
                    torch.cuda.synchronize()
                    t0 = time.time()
                    opt.zero_grad(set_to_none=True)
                    out = model(x)
                    loss = ce(out["pred"], y)
                    loss.backward()
                    opt.step()
                    torch.cuda.synchronize()
                    times.append(time.time() - t0)
                peak = torch.cuda.max_memory_allocated() / 1e9
                print("   batch %3d: peak %.2f GB, %.3f s/step" % (bs, peak, float(np.median(times))))
                SUMMARY["memory"][bs] = {"peak_gb": round(peak, 2), "s_per_step": round(float(np.median(times)), 3)}
            except torch.cuda.OutOfMemoryError:
                print("   batch %3d: OUT OF MEMORY" % bs)
                SUMMARY["memory"][bs] = "OOM"
                break
            finally:
                opt.zero_grad(set_to_none=True)
        print("   (weights were changed by these test steps; nothing is saved)")

    finish(args)


def finish(args):
    SUMMARY["fails"] = FAILS
    if args.log:
        os.makedirs(os.path.dirname(os.path.abspath(args.log)), exist_ok=True)
        with open(args.log, "w") as f:
            json.dump(SUMMARY, f, indent=1)
    print("\n" + ("ALL STAGE 2 CHECKS PASSED" if not FAILS else
                  "%d CHECK(S) FAILED:\n  - " % len(FAILS) + "\n  - ".join(FAILS)))
    sys.exit(1 if FAILS else 0)


if __name__ == "__main__":
    main()


### Cell 5: get DNAFormer repo (same pinned commit as Stage 1)

In [ ]:
!rm -rf {REPO}
!git clone -q https://github.com/itaiorr/Deep-DNA-based-storage {REPO}
!git -C {REPO} checkout -q 167a68261719ea458e79c40fcc485ae768c1a64a
!git -C {REPO} rev-parse HEAD

### Cell 6: download the pretrained checkpoint to Drive (only the first time)
From the official Google Drive link in the DNAFormer README. If this cell fails, see the note at the bottom.

In [ ]:
import gdown, glob
URL = 'https://drive.google.com/drive/folders/1y3cJ3bJdRcrzmEpzlKIosl-b8N5NXEWJ'
if os.path.isfile(CKPT):
    print('Checkpoint already on Drive:', CKPT)
else:
    files = gdown.download_folder(URL, output='/content/dnaf_weights', skip_download=True,
                                  quiet=True, remaining_ok=True)
    pth = [f for f in files if f.path.endswith('DNAFormer_siamese.pth')]
    print('Found:', [f.path for f in pth])
    gdown.download(id=pth[0].id, output=CKPT, quiet=False)
print('Size: %.0f MB' % (os.path.getsize(CKPT) / 1e6))

### Cell 7: run the compatibility check. Paste the whole output of this cell back to Claude.

In [ ]:
!cd {CODE} && python stage2_check.py --repo {REPO} --data {DATA} --ckpt {CKPT} --n-sanity 50 --log {PROJECT}/logs/stage2_check.json 2>&1 | grep -v Warning | tee {PROJECT}/logs/stage2_check.txt

---
**If Cell 6 fails** (Google Drive download limits sometimes block gdown): open the [DNAFormer weights folder](https://drive.google.com/drive/folders/1y3cJ3bJdRcrzmEpzlKIosl-b8N5NXEWJ) in your browser, open `checkpoints`, download `DNAFormer_siamese.pth`, and upload it to **MyDrive > fyp_dna > checkpoints**. Then run Cell 7.